# Demand Planning — Phase 3B Exploration
Auto Components Client | 20 SKUs × 36 Months

**What this notebook covers:**
1. Synthetic dataset — 20 auto-component SKUs × 3 years of monthly actuals
2. Polars — fast in-process DataFrame manipulation
3. DuckDB — SQL-based aggregation and rollups
4. StatsForecast (Nixtla) — AutoETS & AutoARIMA on multiple SKUs
5. HierarchicalForecast — MinTrace reconciliation across Product hierarchy


---
## 1. Synthetic Dataset — 20 Auto Component SKUs × 36 Months

In [ ]:
import numpy as np
import pandas as pd
import polars as pl
import duckdb
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ── Product Hierarchy ─────────────────────────────────────────────────────────
# Category → Sub-category → SKU  (mirrors your product_planning Taxonomy)

PRODUCTS = [
    # (sku_code, name, subcategory, category, base_qty, trend, seasonality_amp)
    # BRAKING SYSTEMS
    ('SKU-001', 'Disc Brake Pad Set — Front',     'Brake Pads',    'Braking Systems',   320, 0.8,  0.20),
    ('SKU-002', 'Disc Brake Pad Set — Rear',      'Brake Pads',    'Braking Systems',   280, 0.5,  0.18),
    ('SKU-003', 'Drum Brake Shoe Set',             'Brake Shoes',   'Braking Systems',   180, 0.2,  0.15),
    ('SKU-004', 'Brake Disc Rotor — Front',        'Brake Rotors',  'Braking Systems',   210, 0.6,  0.12),
    ('SKU-005', 'Brake Caliper Assembly',          'Brake Rotors',  'Braking Systems',    95, 1.0,  0.08),
    # FILTRATION
    ('SKU-006', 'Engine Oil Filter',               'Oil Filters',   'Filtration',        450, 1.2,  0.10),
    ('SKU-007', 'Air Filter Element',              'Air Filters',   'Filtration',        380, 1.0,  0.22),
    ('SKU-008', 'Fuel Filter Cartridge',           'Fuel Filters',  'Filtration',        260, 0.7,  0.14),
    ('SKU-009', 'Cabin Air Filter (Pollen)',       'Air Filters',   'Filtration',        190, 1.5,  0.30),
    # SUSPENSION & STEERING
    ('SKU-010', 'Front Strut Assembly',            'Struts',        'Suspension',        120, 0.4,  0.08),
    ('SKU-011', 'Rear Shock Absorber',             'Shocks',        'Suspension',        140, 0.3,  0.07),
    ('SKU-012', 'Ball Joint — Lower Front',        'Joints',        'Suspension',        160, 0.6,  0.10),
    ('SKU-013', 'Tie Rod End Assembly',            'Steering',      'Suspension',        130, 0.5,  0.09),
    # ELECTRICAL
    ('SKU-014', 'Alternator — 80A',               'Alternators',   'Electrical',         75, 0.9,  0.05),
    ('SKU-015', 'Starter Motor Assembly',          'Starters',      'Electrical',         60, 0.8,  0.04),
    ('SKU-016', 'Ignition Coil Pack',              'Ignition',      'Electrical',        200, 1.1,  0.12),
    # DRIVETRAIN
    ('SKU-017', 'CV Axle Shaft — Front Left',     'CV Axles',      'Drivetrain',        110, 0.7,  0.08),
    ('SKU-018', 'CV Axle Shaft — Front Right',    'CV Axles',      'Drivetrain',        108, 0.7,  0.08),
    ('SKU-019', 'Timing Chain Kit',               'Engine Parts',  'Drivetrain',         55, 0.5,  0.06),
    ('SKU-020', 'Serpentine Belt',                'Engine Parts',  'Drivetrain',        310, 1.3,  0.16),
]

# ── Date spine: Jan 2022 – Dec 2024 ──────────────────────────────────────────
months = pd.date_range('2022-01-01', periods=36, freq='MS')
month_idx = np.arange(36)           # 0..35
month_num  = months.month.values    # 1..12 for seasonality

# ── Generate rows ────────────────────────────────────────────────────────────
rows = []
for sku_code, name, subcat, cat, base, trend_monthly, seas_amp in PRODUCTS:
    for i, ds in enumerate(months):
        trend_factor   = 1 + (trend_monthly / 100) * i
        # Seasonality: peak in Mar-Apr (quarter-end service surge) and Oct-Nov (winter prep)
        seas_angle     = 2 * np.pi * month_num[i] / 12
        seas_factor    = 1 + seas_amp * np.sin(seas_angle - np.pi / 6)
        noise          = np.random.normal(1.0, 0.06)
        qty            = max(1, int(base * trend_factor * seas_factor * noise))
        rows.append({
            'ds'          : ds,
            'sku_code'    : sku_code,
            'sku_name'    : name,
            'subcategory' : subcat,
            'category'    : cat,
            'qty'         : qty,
            'unit_price'  : round(np.random.uniform(180, 4500), 2),  # INR
        })

df_pd = pd.DataFrame(rows)
df_pd['revenue'] = (df_pd['qty'] * df_pd['unit_price']).round(2)

print(f"Dataset shape: {df_pd.shape}")
print(f"Date range  : {df_pd['ds'].min().date()} → {df_pd['ds'].max().date()}")
df_pd.head(10)

In [ ]:
# Quick pivot — monthly qty per SKU (first 6 SKUs)
pivot = df_pd.pivot_table(index='ds', columns='sku_code', values='qty')
print("Monthly Qty — first 6 SKUs")
pivot.iloc[:, :6].to_string()

---
## 2. Polars — Fast In-Process DataFrames
Polars will be your workhorse for building the SKU × Customer × Month tensor before feeding StatsForecast.

In [ ]:
# Convert to Polars
df = pl.from_pandas(df_pd)
print(df.schema)
df.head(5)

In [ ]:
# ── Category-level monthly rollup ─────────────────────────────────────────────
cat_monthly = (
    df
    .group_by(['category', 'ds'])
    .agg([
        pl.sum('qty').alias('total_qty'),
        pl.sum('revenue').alias('total_revenue'),
    ])
    .sort(['category', 'ds'])
)
print("Category × Month rollup:")
cat_monthly.head(12)

In [ ]:
# ── 12-month trailing average per SKU (useful as a naive baseline) ────────────
trailing_avg = (
    df
    .sort('ds')
    .group_by('sku_code')
    .agg([
        pl.col('qty').tail(12).mean().alias('trailing_12m_avg_qty'),
        pl.col('revenue').tail(12).mean().alias('trailing_12m_avg_rev'),
        pl.col('sku_name').first(),
        pl.col('category').first(),
    ])
    .sort('trailing_12m_avg_qty', descending=True)
)
print("Trailing 12-month avg qty per SKU (descending):")
trailing_avg

In [ ]:
# ── Year-over-year growth by category ─────────────────────────────────────────
df_with_year = df.with_columns(pl.col('ds').dt.year().alias('year'))

yoy = (
    df_with_year
    .group_by(['category', 'year'])
    .agg(pl.sum('qty').alias('total_qty'))
    .sort(['category', 'year'])
    .with_columns(
        pl.col('total_qty').shift(1).over('category').alias('prev_year_qty')
    )
    .with_columns(
        ((pl.col('total_qty') - pl.col('prev_year_qty')) / pl.col('prev_year_qty') * 100)
        .round(1)
        .alias('yoy_growth_pct')
    )
    .filter(pl.col('year') > 2022)
)
print("YoY Growth % by Category:")
yoy

---
## 3. DuckDB — SQL Rollups Without Loading Full Tables
In your Django Celery tasks, DuckDB will query Postgres-exported Parquet files (or in-memory frames) for hierarchical aggregation.

In [ ]:
# Register the Polars frame directly — zero-copy via Arrow
con = duckdb.connect()
con.register('actuals', df.to_arrow())

# ── Q1: Best-selling SKU per category (full 3-year period) ────────────────────
print("=== Best-selling SKU per category ===")
con.execute("""
    SELECT category, sku_code, sku_name, SUM(qty) AS total_qty
    FROM actuals
    GROUP BY category, sku_code, sku_name
    QUALIFY RANK() OVER (PARTITION BY category ORDER BY SUM(qty) DESC) = 1
    ORDER BY total_qty DESC
""").df()

In [ ]:
# ── Q2: Monthly seasonality index per SKU (avg_month / overall_avg) ──────────
print("=== Seasonality Index — SKU-001 ===")
con.execute("""
    WITH monthly_avg AS (
        SELECT sku_code,
               MONTH(ds) AS month_num,
               AVG(qty)  AS avg_qty_month
        FROM actuals
        WHERE sku_code = 'SKU-001'
        GROUP BY sku_code, MONTH(ds)
    ),
    overall AS (
        SELECT sku_code, AVG(qty) AS avg_qty_overall
        FROM actuals
        WHERE sku_code = 'SKU-001'
        GROUP BY sku_code
    )
    SELECT m.month_num,
           ROUND(m.avg_qty_month, 1)                                    AS avg_qty,
           ROUND(m.avg_qty_month / o.avg_qty_overall, 3)                AS seasonality_index
    FROM monthly_avg m JOIN overall o USING (sku_code)
    ORDER BY month_num
""").df()

In [ ]:
# ── Q3: Hierarchical rollup (GROUPING SETS) in one SQL pass ───────────────────
# This is the kind of query you'll run in Celery tasks before feeding StatsForecast
print("=== Hierarchical Rollup (Category / Subcategory / SKU) for 2024 ===")
con.execute("""
    SELECT
        COALESCE(category,    '(ALL)') AS category,
        COALESCE(subcategory, '(ALL)') AS subcategory,
        COALESCE(sku_code,    '(ALL)') AS sku_code,
        SUM(qty)     AS total_qty,
        ROUND(SUM(revenue), 0) AS total_revenue_inr
    FROM actuals
    WHERE YEAR(ds) = 2024
    GROUP BY GROUPING SETS (
        (),
        (category),
        (category, subcategory),
        (category, subcategory, sku_code)
    )
    ORDER BY category, subcategory, sku_code
""").df()

---
## 4. StatsForecast — AutoETS & AutoARIMA on Multiple SKUs
StatsForecast's key advantage: it fits **all 20 SKUs in one vectorised call** via Numba, not a Python loop.

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, AutoARIMA, CrostonSBA, SeasonalNaive
from statsforecast.utils import ConformalIntervals

# StatsForecast expects a DataFrame with columns: unique_id, ds, y
sf_df = df_pd[['sku_code', 'ds', 'qty']].rename(columns={'sku_code': 'unique_id', 'qty': 'y'})
sf_df['ds'] = pd.to_datetime(sf_df['ds'])

print(f"Input shape : {sf_df.shape}")
print(f"Series count: {sf_df['unique_id'].nunique()}")
sf_df.head(6)


In [ ]:
# Fit 4 models on all 20 SKUs simultaneously
# season_length=12 → monthly data with annual seasonality
sf = StatsForecast(
    models=[
        AutoETS(season_length=12),
        AutoARIMA(season_length=12),
        SeasonalNaive(season_length=12),      # simple seasonal baseline
        CrostonSBA(),                          # good for intermittent / lumpy demand
    ],
    freq='MS',          # Month Start
    n_jobs=-1,          # use all CPU cores
)

# forecast 6 months ahead (Jan–Jun 2025) with 80% and 95% prediction intervals
forecast_df = sf.forecast(
    df=sf_df, h=6,
    prediction_intervals=ConformalIntervals(h=6, n_windows=2),
    level=[80, 95]
)
print(f"Forecast shape: {forecast_df.shape}")
forecast_df.head(12)


In [ ]:
# ── Cross-validation — rolling origin, 3 windows of 3 months ─────────────────
# This is how you'd compute MAPE/WMAPE for model selection in your ForecastAccuracy model
cv_df = sf.cross_validation(
    df=sf_df,
    h=3,            # horizon
    n_windows=3,    # 3 rolling origins
    step_size=3,
)

# Compute MAPE per model
models_cols = ['AutoETS', 'AutoARIMA', 'SeasonalNaive', 'CrostonSBA']
mape_rows = []
for model in models_cols:
    if model in cv_df.columns:
        ape = (cv_df[model] - cv_df['y']).abs() / cv_df['y'].replace(0, np.nan) * 100
        mape_rows.append({'model': model, 'MAPE': round(ape.mean(), 2)})

mape_df = pd.DataFrame(mape_rows).sort_values('MAPE')
print("\nModel MAPE comparison (lower is better):")
mape_df

In [ ]:
# ── Forecast for a single SKU with prediction intervals ───────────────────────
import matplotlib.pyplot as plt

sku = 'SKU-006'   # Engine Oil Filter — high volume, clear trend

hist    = sf_df[sf_df['unique_id'] == sku].set_index('ds')
fcast   = forecast_df[forecast_df['unique_id'] == sku].set_index('ds')

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(hist.index, hist['y'], label='Actuals', color='steelblue', linewidth=2)

for model, color in [('AutoETS', 'tomato'), ('AutoARIMA', 'seagreen')]:
    if model in fcast.columns:
        ax.plot(fcast.index, fcast[model], label=f'{model} Forecast',
                color=color, linewidth=1.8, linestyle='--')
        lo, hi = f'{model}-lo-80', f'{model}-hi-80'
        if lo in fcast.columns:
            ax.fill_between(fcast.index, fcast[lo], fcast[hi],
                            alpha=0.15, color=color, label=f'{model} 80% CI')

ax.axvline(pd.Timestamp('2025-01-01'), color='gray', linestyle=':', linewidth=1)
ax.set_title(f'StatsForecast — {sku} Engine Oil Filter | Actuals + 6-month Forecast')
ax.set_xlabel('Month'); ax.set_ylabel('Qty')
ax.legend(loc='upper left'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 5. HierarchicalForecast — MinTrace Reconciliation
This is the core of your demand planning module: forecasts at SKU level must **add up correctly** to subcategory and category forecasts. MinTrace ensures that mathematically.

In [ ]:
from hierarchicalforecast.utils import aggregate
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp, MinTrace

# Build the hierarchy spec: list of [level] lists top-down
# Mirrors your Taxonomy: category → subcategory → sku_code
hier_spec = [
    ['category'],
    ['category', 'subcategory'],
    ['category', 'subcategory', 'sku_code'],
]

# aggregate() needs: ds, [hierarchy cols], y (target named 'y')
raw = df_pd[['ds', 'category', 'subcategory', 'sku_code', 'qty']].copy()
raw = raw.rename(columns={'qty': 'y'})
raw['ds'] = pd.to_datetime(raw['ds'])
raw['y'] = raw['y'].astype(float)

# aggregate() returns:
#  Y_df : long DataFrame (unique_id, ds, y) covering ALL hierarchy nodes
#  S_df : summing matrix — maps bottom-level SKUs to all aggregates
#  tags : dict mapping level name → array of node names at that level
Y_df, S_df, tags = aggregate(raw, hier_spec)

print(f"Y_df shape (all nodes × 36 months, long): {Y_df.shape}")
print(f"S_df shape (all nodes × leaf SKUs):        {S_df.shape}")
print(f"Hierarchy levels: {list(tags.keys())}")
print(f"Sample nodes: {Y_df['unique_id'].unique()[:8].tolist()}")
S_df.head()


In [ ]:
# Y_df from aggregate() is already in long format (unique_id, ds, y)
# It covers ALL nodes: top-level categories, subcategories, and leaf SKUs
Y_long = Y_df.copy()

print(f"Long format shape: {Y_long.shape}")
print(f"Total series (nodes): {Y_long['unique_id'].nunique()}")
Y_long.head(8)


In [ ]:
# Fit StatsForecast on ALL hierarchy nodes (SKUs + aggregates)
sf_hier = StatsForecast(
    models=[AutoETS(season_length=12), SeasonalNaive(season_length=12)],
    freq='MS',
    n_jobs=-1,
)

# Forecasts WITH prediction intervals — needed for MinTrace WLS residual weighting
forecasts_df = sf_hier.forecast(
    df=Y_long, h=6,
    fitted=True,
    prediction_intervals=ConformalIntervals(h=6, n_windows=2),
    level=[80, 95]
)
print(f"Base forecasts shape: {forecasts_df.shape}")
forecasts_df.head(6)


In [ ]:
# In-sample fitted values — used by MinTrace to estimate residual variance per node
fitted_df = sf_hier.forecast_fitted_values()
print(f"Fitted values shape: {fitted_df.shape}")
fitted_df.head(4)


In [ ]:
# ── MinTrace Reconciliation ───────────────────────────────────────────────────
# MinTrace(method='ols')      → Ordinary Least Squares
# MinTrace(method='wls_struct') → Weighted by hierarchy structure (good default)
# BottomUp()                  → Simply aggregate SKU forecasts upward (no revision)

hrec = HierarchicalReconciliation(reconcilers=[
    BottomUp(),
    MinTrace(method='ols'),
    MinTrace(method='wls_struct'),
])

reconciled_df = hrec.reconcile(
    Y_hat_df=forecasts_df,
    Y_df=Y_long,
    S_df=S_df,      # note: param is S_df not S in hierarchicalforecast v1.5+
    tags=tags,
)

print(f"Reconciled forecasts shape: {reconciled_df.shape}")
# Columns are suffixed: AutoETS/BottomUp, AutoETS/MinTrace(ols), etc.
reconciled_df.head(6)


In [ ]:
# ── Coherence check: do reconciled SKU forecasts add up to category? ──────────
# Pick AutoETS + MinTrace(ols) column
col = [c for c in reconciled_df.columns if 'MinTrace' in c and 'AutoETS' in c][0]

jan25 = reconciled_df[reconciled_df['ds'] == '2025-01-01'].set_index('unique_id')

braking_skus   = [t for t in jan25.index if 'Braking Systems' in t and '/' not in t.replace('Braking Systems/', '')]
braking_sub_sum = jan25.loc[[t for t in jan25.index if t.startswith('Braking Systems/') and t.count('/') == 1], col].sum()
braking_top     = jan25.loc['Braking Systems', col] if 'Braking Systems' in jan25.index else None

print(f"Jan-2025 Braking Systems — top-level reconciled: {braking_top:.1f}")
print(f"Jan-2025 Braking Systems — sum of sub-categories: {braking_sub_sum:.1f}")
print("✓ Coherent!" if abs(braking_top - braking_sub_sum) < 1 else "✗ Incoherent — check reconciler")

In [ ]:
# ── Visual: Raw vs Reconciled forecast at category level ─────────────────────
raw_col  = [c for c in forecasts_df.columns if 'AutoETS' in c and 'lo' not in c and 'hi' not in c][0]
rec_col  = col  # MinTrace OLS

cat_node  = 'Braking Systems'
hist_node = Y_long[Y_long['unique_id'] == cat_node].set_index('ds')['y']
raw_fcast = forecasts_df[forecasts_df['unique_id'] == cat_node].set_index('ds')[raw_col]
rec_fcast = reconciled_df[reconciled_df['unique_id'] == cat_node].set_index('ds')[rec_col]

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(hist_node, label='Actuals', color='steelblue', linewidth=2)
ax.plot(raw_fcast, label='AutoETS (base, unreconciled)', color='orange',  linestyle='--', linewidth=1.8)
ax.plot(rec_fcast, label='AutoETS + MinTrace(OLS)',      color='seagreen', linestyle='--', linewidth=1.8)
ax.axvline(pd.Timestamp('2025-01-01'), color='gray', linestyle=':', linewidth=1)
ax.set_title(f'HierarchicalForecast — {cat_node} | Raw vs MinTrace-Reconciled')
ax.set_xlabel('Month'); ax.set_ylabel('Qty (Category Total)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ── MinTrace Reconciliation ───────────────────────────────────────────────────
# MinTrace(method='ols')      → Ordinary Least Squares
# MinTrace(method='wls_struct') → Weighted by hierarchy structure (good default)
# BottomUp()                  → Simply aggregate SKU forecasts upward (no revision)

hrec = HierarchicalReconciliation(reconcilers=[
    BottomUp(),
    MinTrace(method='ols'),
    MinTrace(method='wls_struct'),
])

reconciled_df = hrec.reconcile(
    Y_hat_df=forecasts_df,
    Y_df=Y_long,
    S_df=S_df,      # note: param is S_df not S in hierarchicalforecast v1.5+
    tags=tags,
)

print(f"Reconciled forecasts shape: {reconciled_df.shape}")
# Columns are suffixed: AutoETS/BottomUp, AutoETS/MinTrace(ols), etc.
reconciled_df.head(6)
